# Day 4

## Tokenizing with code

In [2]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi my name is Shubham and I like talking to strangers")

In [3]:
tokens

[12194, 922, 1308, 382, 1955, 431, 6595, 326, 357, 1299, 11695, 316, 62511]

In [4]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

12194 = Hi
922 =  my
1308 =  name
382 =  is
1955 =  Sh
431 = ub
6595 = ham
326 =  and
357 =  I
1299 =  like
11695 =  talking
316 =  to
62511 =  strangers


In [10]:
encoding.decode([326])

encoded_txt = encoding.encode("shubham")
for token_id in encoded_txt:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

1116 = sh
431 = ub
6595 = ham


# And another topic!

### The Illusion of "memory"

Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

In [11]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!


### You should be very comfortable with what the next cell is doing!

_I'm creating a new instance of the OpenAI Python Client library, a lightweight wrapper around making HTTP calls to an endpoint for calling the GPT LLM, or other LLM providers_

In [25]:
from openai import OpenAI

model="llama3.2"
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

### A message to OpenAI is a list of dicts

In [26]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Shubham!"}
    ]

In [27]:
response = openai.chat.completions.create(model=model, messages=messages)
response.choices[0].message.content

"Hello Shubham! It's nice to meet you. How's your day going so far? Is there anything I can help you with or would you like to chat for a bit?"

### OK let's now ask a follow-up question

In [28]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [29]:
response = openai.chat.completions.create(model=model, messages=messages)
response.choices[0].message.content

"I don't think I know your name. This is the start of our conversation, and we haven't established any information about you yet. Would you like to share your name with me?"

### Wait, wha??

We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [33]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Shubham!"},
    {"role": "assistant", "content": "Hello Shubham! It's nice to meet you. How's your day going so far? Is there anything I can help you with or would you like to chat for a bit?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [34]:
response = openai.chat.completions.create(model=model, messages=messages)
response.choices[0].message.content

'Your name is Shubham, and we were just chatting earlier. You initially said hello to me as "Hi! I\'m Shubham!"'

## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

